In [ ]:
!pip install timm pillow

In [2]:
import pandas as pd
import numpy as np
import requests
from PIL import Image
from io import BytesIO

from tqdm import tqdm

import torch
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [ ]:
train_df = pd.read_csv("/content/train.csv")

train_df.shape

In [ ]:
sample_df = train_df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

sample_df.shape

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    sample_df["image_link"],
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

In [ ]:
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=0
)

model.eval()

In [7]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [8]:
def get_image_embedding(url):
    try:
        response = requests.get(url, timeout=10)

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image = transform(image)

        image = image.unsqueeze(0)

        with torch.no_grad():
            embedding = model(image)

        return embedding.squeeze().numpy()

    except:
        return np.zeros(2048)

In [ ]:
train_embeddings = []

for url in tqdm(X_train):
    train_embeddings.append(
        get_image_embedding(url)
    )

train_embeddings = np.array(
    train_embeddings
)

train_embeddings.shape

In [ ]:
valid_embeddings = []

for url in tqdm(X_valid):
    valid_embeddings.append(
        get_image_embedding(url)
    )

valid_embeddings = np.array(
    valid_embeddings
)

valid_embeddings.shape

In [ ]:
image_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

image_model.fit(
    train_embeddings,
    y_train
)

In [ ]:
image_predictions = image_model.predict(
    valid_embeddings
)

In [13]:
mae = mean_absolute_error(
    y_valid,
    image_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        image_predictions
    )
)

r2 = r2_score(
    y_valid,
    image_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 17.92264299860102
RMSE: 33.6682938531385
R2 Score: 0.06862659520423964
